# Evaluation

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from config import (
  CLUSTERED_REGENCIES_CSV,
  FEATURE_SELECTION_JSON
)

In [ ]:
TARGET_SILHOUETTE_MIN = 0.50

In [ ]:
print(f"Parameter TARGET_SILHOUETTE_MIN : {TARGET_SILHOUETTE_MIN}")

In [ ]:
df_clustered = pd.read_csv(CLUSTERED_REGENCIES_CSV)

if os.path.exists(FEATURE_SELECTION_JSON):
  with open(FEATURE_SELECTION_JSON, 'r', encoding='utf-8') as f:
    feat_config = json.load(f)
  scaled_cols = feat_config.get('scaled_feature_columns', [c for c in df_clustered.columns if c.startswith('scaled_')])
else:
  scaled_cols = [c for c in df_clustered.columns if c.startswith('scaled_')]

print(f"Total Baris Terklaster : {len(df_clustered)}")
print(f"Fitur Terstandarisasi  : {scaled_cols}")

## Metrik Validasi Internal Klaster

In [ ]:
X_scaled = df_clustered[scaled_cols].values
labels = df_clustered['cluster_label'].values

sil_score = round(float(silhouette_score(X_scaled, labels)), 4)
ch_score = round(float(calinski_harabasz_score(X_scaled, labels)), 2)
db_score = round(float(davies_bouldin_score(X_scaled, labels)), 4)
num_clusters = len(np.unique(labels))

eval_df = pd.DataFrame({
  'Metrik Validasi': [
    'Jumlah Klaster (K)',
    'Silhouette Coefficient',
    'Calinski-Harabasz Index',
    'Davies-Bouldin Index'
  ],
  'Nilai Evaluasi': [
    num_clusters,
    sil_score,
    ch_score,
    db_score
  ],
  'Kriteria / Target': [
    'Optimal Kneedle',
    f'>= {TARGET_SILHOUETTE_MIN} (Kerapatan Klaster)',
    'Semakin Tinggi Semakin Baik',
    'Semakin Rendah Semakin Baik'
  ]
})
print(eval_df.to_markdown(index=False))

## Eksplorasi Visual Hasil Klasterisasi

### Proyeksi 2D Klaster (PCA)

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
df_pca = pd.DataFrame(X_pca, columns=['PCA1', 'PCA2'])
df_pca['cluster'] = [f"Klaster {lbl}" for lbl in labels]

plt.figure(figsize=(8, 6))
sns.scatterplot(x='PCA1', y='PCA2', hue='cluster', data=df_pca, palette='tab10', s=60, alpha=0.85)
plt.title('Proyeksi 2D Klasterisasi (Principal Component Analysis)')
plt.tight_layout()
plt.show()

### Distribusi Jumlah Anggota per Klaster

In [ ]:
cluster_dist = df_clustered['cluster_label'].value_counts().sort_index()
dist_df = pd.DataFrame({
  'Klaster': [f"Klaster {c}" for c in cluster_dist.index],
  'Jumlah Kab/Kota': cluster_dist.values,
  'Persentase (%)': (cluster_dist.values / len(df_clustered) * 100).round(2)
})
print(dist_df.to_markdown(index=False))